# Fine-tune Qwen2.5-VL-3B-Instruct với Unsloth QLoRA

**Target platform:** Kaggle T4 | Colab | Local RTX 5060 Ti

**Data sources (upload lên Kaggle Dataset tên `a20-finetune-data`):**
- `question_bank.jsonl` — 1,276 MCQs (CS224n, CS231n, CS230)
- `units.jsonl` — 376 learning units
- `qa_history.jsonl` — 58 production Q&A (optional)

**Strategy:** MCQ → 3 ChatML variants (~3,800 samples) + unit summaries (~376 samples)

**Split:** Theo `lecture_id` để tránh data leakage

---
### Kaggle setup (làm 1 lần)
1. Vào `kaggle.com/datasets` → New Dataset → upload 3 files → đặt tên `a20-finetune-data`
2. Trong notebook: **+ Add Data** → tìm `a20-finetune-data` → Add
3. Settings → Accelerator → **GPU T4 × 1**
4. Run All

## 0. Environment detection & Install

In [ ]:
import os
import sys
from pathlib import Path

if os.path.exists("/kaggle/working"):
    ENV = "kaggle"
elif os.path.exists("/content"):
    ENV = "colab"
else:
    ENV = "local"

print(f"Environment: {ENV}")

In [ ]:
if ENV == "local":
    os.system('pip install "unsloth[cu128] @ git+https://github.com/unslothai/unsloth.git" -q')
else:
    os.system('pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q')

os.system('pip install transformers datasets accelerate trl bitsandbytes matplotlib -q')

In [ ]:
import torch

print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

# Ampere+ (sm>=80) mới support bf16 hardware — T4 là sm_75, phải dùng fp16
major, _ = torch.cuda.get_device_capability(0)
USE_BF16  = major >= 8
USE_FP16  = not USE_BF16
print(f"Compute capability: sm_{major}x → precision: {'bf16' if USE_BF16 else 'fp16'}")

## 1. Paths & Load Data

In [ ]:
import json

if ENV == "kaggle":
    DATA_ROOT = Path("/kaggle/input/a20-finetune-data")
    OUT_ROOT  = Path("/kaggle/working")
elif ENV == "colab":
    DATA_ROOT = Path("/content/drive/MyDrive/a20-data")
    OUT_ROOT  = Path("/content/outputs")
else:
    DATA_ROOT = Path("../data/final_artifacts/cs224n_cs231n_cs230_v1/canonical")
    OUT_ROOT  = Path(".")

QUESTION_BANK = DATA_ROOT / "question_bank.jsonl"
UNITS_FILE    = DATA_ROOT / "units.jsonl"
QA_HISTORY    = DATA_ROOT / "qa_history.jsonl"
OUTPUT_DIR    = OUT_ROOT / "data/sft"
CKPT_DIR      = OUT_ROOT / "checkpoints/tutor-vl3b-v1"
MODEL_DIR     = OUT_ROOT / "models/tutor-vl3b-v1-merged"

# Kiểm tra input files tồn tại trước khi chạy
assert QUESTION_BANK.exists(), f"Missing: {QUESTION_BANK}"
assert UNITS_FILE.exists(),    f"Missing: {UNITS_FILE}"
print("Input files OK")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

mcqs   = load_jsonl(QUESTION_BANK)
units  = load_jsonl(UNITS_FILE)
qa_log = load_jsonl(QA_HISTORY) if QA_HISTORY.exists() else []

print(f"MCQs:   {len(mcqs)}")
print(f"Units:  {len(units)}")
print(f"QA log: {len(qa_log)}")
print("\nMCQ sample:")
print(json.dumps(mcqs[0], indent=2, ensure_ascii=False))

In [ ]:
from collections import Counter

course_dist   = Counter(m["course_id"] for m in mcqs)
lecture_count = len(set(m["lecture_id"] for m in mcqs))

print("Course distribution:")
for course, count in course_dist.most_common():
    print(f"  {course}: {count}")
print(f"Unique lectures: {lecture_count}")

## 2. Convert MCQ → ChatML

3 variants per MCQ:
- **V1 (direct):** Hỏi đáp án + giải thích
- **V2 (elaboration):** Mở rộng khái niệm (chỉ khi có explanation)
- **V3 (distractor):** Phân tích tại sao đáp án sai (chỉ khi có explanation)

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert AI tutor for graduate-level ML courses "
    "(CS224n NLP, CS231n Computer Vision, CS230 Deep Learning). "
    "Answer concisely and educationally. Ground your answers in course material when possible."
)

def mcq_to_variants(item):
    q       = item["question"]
    choice  = item["choices"][item["answer_index"]]
    expl    = item.get("explanation", "")
    course  = item["course_id"]
    wrong   = [c for i, c in enumerate(item["choices"]) if i != item["answer_index"]]
    lecture = item.get("lecture_id", "")
    base    = {"lecture_id": lecture, "item_id": item["item_id"]}

    v1 = {
        **base, "variant": "v1",
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": f"[{course}] {q}\n\nChoices:\n" + "\n".join(f"- {c}" for c in item["choices"])},
            {"role": "assistant", "content": f"The correct answer is: **{choice}**\n\n{expl}"},
        ]
    }

    v2 = None
    if expl:
        v2 = {
            **base, "variant": "v2",
            "messages": [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": f"[{course}] Explain the concept behind this question: {q}"},
                {"role": "assistant", "content": f"{expl}\n\nIn summary, the correct answer is: {choice}"},
            ]
        }

    v3 = None
    if wrong and expl:
        v3 = {
            **base, "variant": "v3",
            "messages": [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": f"[{course}] Why is '{wrong[0]}' not the correct answer to: {q}"},
                {"role": "assistant", "content": f"'{wrong[0]}' is incorrect. {expl} Therefore, the correct answer is '{choice}'."},
            ]
        }

    return [v for v in [v1, v2, v3] if v is not None]


def unit_to_sample(unit):
    title   = unit.get("title", "")
    summary = unit.get("summary", "") or unit.get("description", "")
    course  = unit.get("course_id", "")
    lecture = unit.get("lecture_id", "")
    if not title or not summary or len(summary) < 50:
        return None
    return {
        "lecture_id": lecture,
        "item_id": unit.get("unit_id", unit.get("id", "")),
        "variant": "unit_summary",
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": f"[{course}] Summarize the key concepts in '{title}'"},
            {"role": "assistant", "content": summary},
        ]
    }


all_samples = []
for item in mcqs:
    all_samples.extend(mcq_to_variants(item))

unit_samples = [s for u in units if (s := unit_to_sample(u)) is not None]
all_samples.extend(unit_samples)

variant_counts = Counter(s["variant"] for s in all_samples)
print("Variant distribution:")
for v, c in variant_counts.most_common():
    print(f"  {v}: {c}")
print(f"Total: {len(all_samples)}")

## 3. Split by lecture_id (90/5/5)

In [ ]:
import random
from collections import defaultdict

random.seed(42)

lecture_groups = defaultdict(list)
for s in all_samples:
    lecture_groups[s["lecture_id"]].append(s)

lectures = list(lecture_groups.keys())
random.shuffle(lectures)

n      = len(lectures)
n_val  = max(1, int(n * 0.05))
n_test = max(1, int(n * 0.05))

val_lectures   = set(lectures[:n_val])
test_lectures  = set(lectures[n_val:n_val + n_test])
train_lectures = set(lectures[n_val + n_test:])

train = [s for lid in train_lectures for s in lecture_groups[lid]]
val   = [s for lid in val_lectures   for s in lecture_groups[lid]]
test  = [s for lid in test_lectures  for s in lecture_groups[lid]]

print(f"Lectures: {len(train_lectures)} train / {len(val_lectures)} val / {len(test_lectures)} test")
print(f"Samples:  {len(train)} train / {len(val)} val / {len(test)} test")

def save_jsonl(data, path):
    with open(path, "w") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"Saved {len(data)} → {path}")

save_jsonl(train, OUTPUT_DIR / "train.jsonl")
save_jsonl(val,   OUTPUT_DIR / "val.jsonl")
save_jsonl(test,  OUTPUT_DIR / "test.jsonl")

## 4. Fine-tune với Unsloth

### 4a. Load model

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    model_name="unsloth/Qwen2.5-VL-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

total_params = sum(p.numel() for p in model.parameters()) / 1e9
used_vram    = torch.cuda.memory_allocated(0) / 1e9
print(f"Model: {total_params:.2f}B params | VRAM used: {used_vram:.1f} GB")

### 4b. LoRA setup (vision frozen)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable / 1e6:.1f}M / {total / 1e6:.1f}M ({100 * trainable / total:.1f}%)")

### 4c. Prepare dataset

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def format_sample(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )}

def load_hf_dataset(path):
    data = load_jsonl(path)
    return Dataset.from_list([{"messages": d["messages"]} for d in data]).map(format_sample)

train_dataset = load_hf_dataset(OUTPUT_DIR / "train.jsonl")
val_dataset   = load_hf_dataset(OUTPUT_DIR / "val.jsonl")

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print("\nSample:")
print(train_dataset[0]["text"][:400])

### 4d. Overfit gate (optional)

16 samples × 80 steps để verify data format và model load OK.  
Tắt bằng `RUN_OVERFIT_GATE = False` nếu muốn skip (gate làm dirty model weights trước full training).

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

RUN_OVERFIT_GATE = False  # Set True để chạy smoke test trước khi train full

if RUN_OVERFIT_GATE:
    gate_trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset.select(range(16)),
        args=SFTConfig(
            output_dir=str(CKPT_DIR / "gate"),
            per_device_train_batch_size=2,
            gradient_accumulation_steps=1,
            max_steps=80,
            learning_rate=2e-4,
            bf16=USE_BF16,
            fp16=USE_FP16,
            logging_steps=10,
            report_to="none",
            dataset_text_field="text",
            max_seq_length=512,
        ),
    )
    gate_trainer = train_on_responses_only(
        gate_trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    gate_result = gate_trainer.train()
    print(f"Gate loss: {gate_result.training_loss:.4f}")
    assert gate_result.training_loss < 1.0, "GATE FAILED — kiểm tra lại data format!"
    print("GATE PASSED")
else:
    print("Overfit gate skipped (RUN_OVERFIT_GATE=False)")

### 4e. Full training

In [ ]:
FastVisionModel.for_training(model)  # ensure training mode (quan trọng nếu gate đã chạy)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        output_dir=str(CKPT_DIR),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # effective batch = 16
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=50,
        bf16=USE_BF16,
        fp16=USE_FP16,
        optim="adamw_8bit",
        logging_steps=25,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        dataset_text_field="text",
        max_seq_length=2048,
        report_to="none",
        dataloader_num_workers=2,
    ),
)

# Chỉ tính loss trên phần assistant — không tính system/user tokens
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer.train()

ckpts = sorted(CKPT_DIR.glob("checkpoint-*"))
print(f"\nCheckpoints saved ({len(ckpts)}):")
for ckpt in ckpts:
    size = sum(f.stat().st_size for f in ckpt.rglob("*") if f.is_file()) / 1e6
    print(f"  {ckpt.name}  ({size:.0f} MB)")

## 5. Loss Chart

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_logs  = [(e["epoch"], e["loss"])      for e in log_history if "loss"      in e and "eval_loss" not in e]
eval_logs   = [(e["epoch"], e["eval_loss"]) for e in log_history if "eval_loss" in e]

fig, ax = plt.subplots(figsize=(10, 5))

if train_logs:
    ep_t, loss_t = zip(*train_logs)
    ax.plot(ep_t, loss_t, label="Train loss", color="steelblue", linewidth=1.5)

if eval_logs:
    ep_e, loss_e = zip(*eval_logs)
    ax.scatter(ep_e, loss_e, label="Eval loss", color="orange", s=120, zorder=5)
    ax.plot(ep_e, loss_e, color="orange", linestyle="--", linewidth=1)

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Qwen2.5-VL-3B QLoRA — Training & Eval Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

chart_path = OUT_ROOT / "loss_curve.png"
plt.savefig(str(chart_path), dpi=150)
plt.show()

if train_logs:
    print(f"Final train loss: {loss_t[-1]:.4f}")
if eval_logs:
    print(f"Final eval loss:  {loss_e[-1]:.4f}")
    best = min(zip(loss_e, ep_e))
    print(f"Best  eval loss:  {best[0]:.4f}  (epoch {best[1]:.0f})")
print(f"Chart saved → {chart_path}")

## 6. Evaluation trên test set

### Dataset này chứng minh được gì?

| Khía cạnh | Đánh giá |
|---|---|
| **Giải thích khái niệm ML** (CS224n/CS231n/CS230) | ✅ ~4,200 mẫu, đủ bằng chứng |
| **Tutor style** (giải thích thay vì trả lời thẳng) | ✅ V2/V3 variants rèn ngữ điệu |
| **Image/frame support** | ✅ Base VLM giữ nguyên — vision tower frozen, không bị xóa |
| **Visual grounding** (hiểu nội dung trong ảnh) | ⚠️ SFT data là text-only → không cải thiện, dựa vào base model |
| **Conversational / multi-turn** | ❌ Không có — chỉ single-turn Q&A |
| **Tiếng Việt** | ❌ Không có — toàn bộ data là tiếng Anh |
| **Tool calling** | ❌ Không có samples tool-call |

> **Vision note:** Pipeline hiện tại gửi `image_base64` (JPEG frame) vào model. Fine-tune này không làm hỏng vision support vì tower bị freeze — model vẫn xử lý ảnh như base Qwen2.5-VL. Chỉ là khả năng *visual grounding* không được cải thiện qua SFT này.

In [ ]:
FastVisionModel.for_inference(model)

def generate(messages, max_new_tokens=200):
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=max_new_tokens, temperature=0.1, do_sample=True)
    return tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)


test_data   = load_jsonl(OUTPUT_DIR / "test.jsonl")
test_sample = test_data[:5]

for i, sample in enumerate(test_sample):
    user_msg = next(m["content"] for m in sample["messages"] if m["role"] == "user")
    ref_msg  = next(m["content"] for m in sample["messages"] if m["role"] == "assistant")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    response = generate(messages)

    print(f"─── Sample {i+1} [{sample['variant']}] ───")
    print(f"Q:   {user_msg[:120]}")
    print(f"Ref: {ref_msg[:150]}")
    print(f"Gen: {response[:150]}")
    print()

## 7. Save adapter + merge

Output lưu vào `/kaggle/working/` — download từ **Output** tab sau khi notebook chạy xong.

In [ ]:
model.save_pretrained(str(CKPT_DIR))
tokenizer.save_pretrained(str(CKPT_DIR))
print(f"LoRA adapter → {CKPT_DIR}")

In [ ]:
DO_MERGE = False  # Set True để merge LoRA vào base model (cần cho vLLM, ~6GB disk)

if DO_MERGE:
    model.save_pretrained_merged(
        str(MODEL_DIR),
        tokenizer,
        save_method="merged_16bit",
    )
    print(f"Merged model → {MODEL_DIR}")
else:
    print("Merge skipped (DO_MERGE=False) — dùng LoRA adapter để inference hoặc set DO_MERGE=True khi cần vLLM")

import subprocess
if ENV == "kaggle":
    result = subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True)
    print(f"Total output size: {result.stdout.strip()}")
    print("Download: Output tab → Download all")